# ICD 341 – Advanced Statistical Computing
## Labs 5–8

**Name:** Reuben Kambowa Luneta  **Student Number:** 202402454

In [1]:
import numpy as np
import scipy.linalg as sla
from scipy import optimize, sparse, stats
np.set_printoptions(precision=4, suppress=True)

# LAB 5 – Linear Systems and Regression

### Q1. Solve  3x + 2y = 12,  x + 4y = 10

In [2]:
A = np.array([[3., 2.],
              [1., 4.]])
b = np.array([12., 10.])
sol = np.linalg.solve(A, b)
print("x =", sol[0], ", y =", sol[1])
print("Check A @ sol =", A @ sol)

x = 2.8000000000000003 , y = 1.7999999999999998
Check A @ sol = [12. 10.]


### Q2. A 3 × 3 system using `np.linalg.solve()`

Solve: 2x + y − z = 8,  −3x − y + 2z = −11,  −2x + y + 2z = −3

In [3]:
A3 = np.array([[ 2.,  1., -1.],
               [-3., -1.,  2.],
               [-2.,  1.,  2.]])
b3 = np.array([8., -11., -3.])
x3 = np.linalg.solve(A3, b3)
print("Solution (x, y, z):", x3)
print("Residual A@x - b  :", A3 @ x3 - b3)

Solution (x, y, z): [ 2.  3. -1.]
Residual A@x - b  : [-0.  0.  0.]


### Q3. Why `np.linalg.solve(A, b)` is preferred to computing `A⁻¹b`

1. **Speed:** `solve` factorises A once (LU decomposition with partial pivoting, ≈ (2/3)n³ flops) and then does cheap forward/back substitution. Forming the inverse costs about 2n³ flops, and then you still need a matrix–vector product.
2. **Numerical accuracy/stability:** The inverse amplifies rounding errors, especially when A is ill-conditioned. `solve` gives a smaller residual `‖Ax − b‖` in practice because it never forms A⁻¹ explicitly.
3. **Structure and memory:** Inverting destroys sparsity/structure (the inverse of a sparse matrix is generally dense), and it uses more memory.
4. **Detects problems:** `solve` raises `LinAlgError` for singular matrices instead of silently returning garbage.
5. **Multiple right-hand sides:** `solve(A, B)` handles many `b` vectors with the single factorisation.

Demonstration on an ill-conditioned (Hilbert) matrix:

In [4]:
n = 10
H = sla.hilbert(n)
x_true = np.ones(n)
bH = H @ x_true

x_solve = np.linalg.solve(H, bH)
x_inv   = np.linalg.inv(H) @ bH

print("cond(H) = %.3e" % np.linalg.cond(H))
print("Error using solve : %.3e" % np.linalg.norm(x_solve - x_true))
print("Error using inv@b : %.3e" % np.linalg.norm(x_inv   - x_true))

cond(H) = 1.602e+13
Error using solve : 7.185e-05
Error using inv@b : 1.130e-02


### Q4. Consumption (Y) vs disposable income (X) – 10 Lusaka households

In [5]:
X = np.array([30, 45, 60, 100, 150, 200, 240, 250, 300, 320], dtype=float)
Y = np.array([25, 20, 40,  65, 120, 190, 200, 200, 260, 220], dtype=float)

**(i)** Regression design matrix: $\mathbf{X}=[\mathbf{1}\;\; x]$ and $\hat\beta=(\mathbf{X}^T\mathbf{X})^{-1}\mathbf{X}^T\mathbf{y}$, computed by solving the normal equations.

In [6]:
Xd = np.column_stack([np.ones_like(X), X])       # design matrix
print("Design matrix:\n", Xd)

beta = np.linalg.solve(Xd.T @ Xd, Xd.T @ Y)      # normal equations
print("\nbeta0 (intercept) = %.4f" % beta[0])
print("beta1 (slope)     = %.4f" % beta[1])
print("Fitted model: Y_hat = %.4f + %.4f X" % (beta[0], beta[1]))

# cross-check with least squares
print("Check (lstsq):", np.linalg.lstsq(Xd, Y, rcond=None)[0])

Design matrix:
 [[  1.  30.]
 [  1.  45.]
 [  1.  60.]
 [  1. 100.]
 [  1. 150.]
 [  1. 200.]
 [  1. 240.]
 [  1. 250.]
 [  1. 300.]
 [  1. 320.]]

beta0 (intercept) = -5.5715
beta1 (slope)     = 0.8234
Fitted model: Y_hat = -5.5715 + 0.8234 X
Check (lstsq): [-5.5715  0.8234]


**(ii)** Predicted consumption when disposable income = 181

In [7]:
y181 = beta[0] + beta[1] * 181
print("Estimated consumption at X = 181:  %.2f" % y181)

Estimated consumption at X = 181:  143.47


# LAB 6 – Matrix Decompositions

### Q1. Covariance matrix and Cholesky decomposition

In [8]:
rng = np.random.default_rng(341)
data = rng.normal(size=(200, 3)) @ np.array([[1.0, 0.5, 0.2],
                                             [0.0, 1.0, 0.4],
                                             [0.0, 0.0, 1.0]])
S = np.cov(data, rowvar=False)
print("Covariance matrix S:\n", S)

L = np.linalg.cholesky(S)          # S = L L^T, L lower triangular
print("\nCholesky factor L:\n", L)
print("\nL @ L.T equals S?", np.allclose(L @ L.T, S))

Covariance matrix S:
 [[1.1698 0.6026 0.1288]
 [0.6026 1.2041 0.3572]
 [0.1288 0.3572 1.241 ]]

Cholesky factor L:
 [[1.0816 0.     0.    ]
 [0.5572 0.9453 0.    ]
 [0.1191 0.3077 1.064 ]]

L @ L.T equals S? True


### Q2. QR decomposition

In [9]:
M = np.array([[1., 2., 3.],
              [4., 5., 6.],
              [7., 8., 10.],
              [2., 1., 0.]])
Q, R = np.linalg.qr(M)
print("Q (orthonormal columns):\n", Q)
print("\nR (upper triangular):\n", R)
print("\nQ.T @ Q = I?   ", np.allclose(Q.T @ Q, np.eye(3)))
print("Q @ R = M?     ", np.allclose(Q @ R, M))

Q (orthonormal columns):
 [[-0.1195 -0.5345  0.1826]
 [-0.4781 -0.2673  0.7303]
 [-0.8367  0.     -0.5477]
 [-0.239   0.8018  0.3651]]

R (upper triangular):
 [[ -8.3666  -9.5618 -11.5937]
 [  0.      -1.6036  -3.2071]
 [  0.       0.      -0.5477]]

Q.T @ Q = I?    True
Q @ R = M?      True


### Q3. Reconstruct a matrix using SVD

In [10]:
U, s, Vt = np.linalg.svd(M, full_matrices=False)
print("Singular values:", s)
M_rec = U @ np.diag(s) @ Vt
print("\nReconstructed matrix:\n", M_rec)
print("\nReconstruction exact?", np.allclose(M_rec, M))

# bonus: rank-2 approximation
k = 2
M_k = U[:, :k] @ np.diag(s[:k]) @ Vt[:k, :]
print("\nRank-2 approximation error (Frobenius): %.4f" % np.linalg.norm(M - M_k))

Singular values: [17.4761  1.8801  0.2237]

Reconstructed matrix:
 [[ 1.  2.  3.]
 [ 4.  5.  6.]
 [ 7.  8. 10.]
 [ 2.  1.  0.]]

Reconstruction exact? True

Rank-2 approximation error (Frobenius): 0.2237


### Q4. Condition number of a matrix

In [11]:
A_good = np.array([[2., 0.], [0., 1.]])
A_bad  = np.array([[1., 1.], [1., 1.0001]])
print("cond(A_good) = %.4f" % np.linalg.cond(A_good))
print("cond(A_bad)  = %.4e" % np.linalg.cond(A_bad))
print("cond(M) (2-norm) = %.4f  [= s_max/s_min = %.4f]" % (np.linalg.cond(M), s[0]/s[-1]))

cond(A_good) = 2.0000
cond(A_bad)  = 4.0002e+04
cond(M) (2-norm) = 78.1392  [= s_max/s_min = 78.1392]


A large condition number means the matrix is nearly singular: small changes in the data produce large changes in the solution of `Ax = b`, so results lose accuracy.

### Q5. A sparse 1000 × 1000 matrix

In [12]:
n = 1000
S_sp = sparse.random(n, n, density=0.01, format="csr", random_state=341)
print("Shape               :", S_sp.shape)
print("Non-zero entries    :", S_sp.nnz, "of", n*n, "(%.2f%%)" % (100*S_sp.nnz/(n*n)))

# memory comparison
dense = S_sp.toarray()
sparse_bytes = S_sp.data.nbytes + S_sp.indices.nbytes + S_sp.indptr.nbytes
print("Dense memory  : %.2f MB" % (dense.nbytes/1e6))
print("Sparse memory : %.3f MB" % (sparse_bytes/1e6))

# a second example: tridiagonal matrix
T = sparse.diags([np.ones(n-1), -2*np.ones(n), np.ones(n-1)], [-1, 0, 1], format="csr")
print("\nTridiagonal matrix non-zeros:", T.nnz)

Shape               : (1000, 1000)
Non-zero entries    : 10000 of 1000000 (1.00%)
Dense memory  : 8.00 MB
Sparse memory : 0.124 MB

Tridiagonal matrix non-zeros: 2998


### Q6. Why sparse matrices are useful for large datasets

* **Memory:** Only non-zero entries (and their indices) are stored. A 1000 × 1000 matrix with 1% density needs roughly 1% of the dense storage; for 10⁶ × 10⁶ matrices, dense storage is impossible while sparse is feasible.
* **Speed:** Operations (matrix–vector products, solves) skip zeros, so cost scales with the number of non-zeros rather than n².
* **Scalability:** Many real datasets are naturally sparse – text (document-term matrices), networks/graphs, recommender systems, finite-element models, one-hot encoded features.
* **Specialised algorithms:** Iterative solvers and sparse factorisations (`scipy.sparse.linalg`) exploit the structure.

# LAB 7 – Optimisation and MLE

### Q1. Gradient descent for f(x) = (x − 10)²

Gradient: f′(x) = 2(x − 10).

In [13]:
def f(x):      return (x - 10)**2
def grad_f(x): return 2*(x - 10)

def gradient_descent(grad, x0, lr=0.1, tol=1e-8, max_iter=10000):
    x, path = x0, [x0]
    for i in range(max_iter):
        x_new = x - lr * grad(x)
        path.append(x_new)
        if abs(x_new - x) < tol:
            break
        x = x_new
    return x_new, np.array(path), i + 1

x_min, path, iters = gradient_descent(grad_f, x0=0.0, lr=0.1)
print("Minimiser x* = %.6f  (true value 10)" % x_min)
print("f(x*)        = %.2e" % f(x_min))
print("Iterations   =", iters)

Minimiser x* = 10.000000  (true value 10)
f(x*)        = 1.37e-15
Iterations   = 87


### Q2. BFGS vs CG on f(x, y) = (x − 4)² + (y − 6)²

In [14]:
def g(p):      return (p[0]-4)**2 + (p[1]-6)**2
def g_grad(p): return np.array([2*(p[0]-4), 2*(p[1]-6)])

x0 = np.array([0., 0.])
res_bfgs = optimize.minimize(g, x0, jac=g_grad, method="BFGS")
res_cg   = optimize.minimize(g, x0, jac=g_grad, method="CG")

print("%-6s %-22s %-12s %-6s %-6s %-6s" % ("Method", "Minimiser", "f(min)", "nit", "nfev", "njev"))
for name, r in [("BFGS", res_bfgs), ("CG", res_cg)]:
    print("%-6s %-22s %-12.2e %-6d %-6d %-6d" % (name, np.round(r.x, 6), r.fun, r.nit, r.nfev, r.njev))

Method Minimiser              f(min)       nit    nfev   njev  
BFGS   [4. 6.]                1.58e-30     3      4      4     
CG     [4. 6.]                0.00e+00     2      5      5     


Both methods find the minimum at (4, 6). BFGS is a quasi-Newton method: it builds an approximation to the inverse Hessian (O(n²) memory), and typically converges in fewer iterations. CG (nonlinear conjugate gradient) only stores a few vectors (O(n) memory), so it is better suited to very high-dimensional problems. On this simple quadratic both converge in very few iterations.

### Q3. Effect of changing the learning rate

In [15]:
print("%-8s %-12s %-14s %s" % ("lr", "iterations", "final x", "comment"))
for lr in [0.001, 0.01, 0.1, 0.5, 0.9, 1.0, 1.05]:
    x = 0.0
    converged = False
    for i in range(1, 5001):
        x = x - lr*grad_f(x)
        if abs(x) > 1e10:
            break
        if abs(x - 10) < 1e-6:
            converged = True
            break
    comment = "converged" if converged else ("DIVERGED" if abs(x) > 1e10 else ("oscillates forever (error multiplier = -1)" if lr >= 1 else "too slow (not converged in 5000 its)"))
    print("%-8s %-12d %-14.4g %s" % (lr, i, x, comment))

lr       iterations   final x        comment
0.001    5000         10             too slow (not converged in 5000 its)
0.01     798          10             converged
0.1      73           10             converged
0.5      1            10             converged
0.9      73           10             converged
1.0      5000         0              oscillates forever (error multiplier = -1)
1.05     218          -1.056e+10     DIVERGED


**Interpretation.** For f(x) = (x − 10)² the update is x ← x − lr·2(x − 10), so the error is multiplied by (1 − 2·lr) each step.

* **Too small** (e.g. 0.001): converges, but very slowly.
* **Moderate** (e.g. 0.1–0.5): fast convergence. At lr = 0.5 it lands on the minimum in one step (error multiplier = 0).
* **Between 0.5 and 1.0:** the iterates overshoot and oscillate around the minimum but still converge.
* **lr = 1.0:** the error multiplier is −1, so it oscillates forever without converging.
* **lr > 1.0:** |1 − 2·lr| > 1, so the iterates **diverge**.

In general, the learning rate trades off speed against stability and must stay below a threshold set by the curvature (here lr < 2/f″ = 1).

### Q4. MLE of the mean of normal observations (SciPy)

In [16]:
rng = np.random.default_rng(341)
true_mu, true_sigma = 5.0, 2.0
obs = rng.normal(true_mu, true_sigma, size=200)

# negative log-likelihood with sigma treated as known (= 2)
def nll_mu(mu):
    return -np.sum(stats.norm.logpdf(obs, loc=mu[0], scale=true_sigma))

res = optimize.minimize(nll_mu, x0=[0.0], method="BFGS")
print("MLE of mu (optimiser) = %.5f" % res.x[0])
print("Sample mean           = %.5f" % obs.mean())

# joint MLE of (mu, sigma) using log(sigma) for positivity
def nll_both(theta):
    mu, log_s = theta
    return -np.sum(stats.norm.logpdf(obs, loc=mu, scale=np.exp(log_s)))
res2 = optimize.minimize(nll_both, x0=[0.0, 0.0], method="BFGS")
print("\nJoint MLE: mu = %.5f, sigma = %.5f" % (res2.x[0], np.exp(res2.x[1])))
print("Closed form: mean = %.5f, sd (ddof=0) = %.5f" % (obs.mean(), obs.std()))

MLE of mu (optimiser) = 5.03709
Sample mean           = 5.03709

Joint MLE: mu = 5.03709, sigma = 2.10702
Closed form: mean = 5.03709, sd (ddof=0) = 2.10702


The numerical MLE of μ equals the sample mean, as theory predicts.

### Q5. Sigmoid function

In [17]:
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    # numerically stable version
    out = np.empty_like(z)
    pos = z >= 0
    out[pos]  = 1.0 / (1.0 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1.0 + ez)
    return out

print("sigmoid(0)   =", sigmoid(0.0))
print("sigmoid(-5..5) =", sigmoid(np.arange(-5, 6)))
print("sigmoid(+-1000) =", sigmoid(np.array([-1000., 1000.])), "(no overflow)")

sigmoid(0)   = 0.5
sigmoid(-5..5) = [0.0067 0.018  0.0474 0.1192 0.2689 0.5    0.7311 0.8808 0.9526 0.982
 0.9933]
sigmoid(+-1000) = [0. 1.] (no overflow)


### Q6. Simple logistic regression via MLE – study hours vs pass/fail

In [18]:
hours = np.arange(1, 9, dtype=float)
passed = np.array([0, 0, 0, 0, 1, 1, 1, 1], dtype=float)

def neg_loglik(beta, X, y):
    z = beta[0] + beta[1]*X
    # log-likelihood: sum[y*z - log(1+e^z)], computed stably
    return -np.sum(y*z - np.logaddexp(0, z))

def neg_loglik_grad(beta, X, y):
    p = sigmoid(beta[0] + beta[1]*X)
    return -np.array([np.sum(y - p), np.sum((y - p)*X)])

res = optimize.minimize(neg_loglik, x0=[0., 0.], args=(hours, passed),
                        jac=neg_loglik_grad, method="BFGS")
print("MLE attempt: beta0 = %.3f, beta1 = %.3f" % tuple(res.x))
print("Log-likelihood = %.6f   (maximum possible is 0)" % -res.fun)
print("Optimiser message:", res.message)

MLE attempt: beta0 = -127.192, beta1 = 28.047
Log-likelihood = -0.000002   (maximum possible is 0)
Optimiser message: Optimization terminated successfully.


**Important observation – perfect separation.** All students with X ≤ 4 failed and all with X ≥ 5 passed, so the data are *perfectly separable* at X = 4.5. In this situation the likelihood keeps increasing as the slope grows (the fitted curve becomes a step function), so a finite MLE does **not exist**: the optimiser just stops at some large, arbitrary coefficients, and the log-likelihood approaches 0. The *direction* of the solution is still informative: β₁ > 0 and the 50% point is at X = −β₀/β₁ ≈ 4.5.

To get a stable, finite estimate we can add a small ridge (L2) penalty, shown next.

In [19]:
def pen_nll(beta, X, y, lam):
    return neg_loglik(beta, X, y) + 0.5*lam*beta[1]**2

for lam in [0.01, 0.1, 1.0]:
    r = optimize.minimize(pen_nll, x0=[0., 0.], args=(hours, passed, lam), method="BFGS")
    b0, b1 = r.x
    print("lambda=%-5s beta0=%8.4f beta1=%7.4f  P(pass | 4.5 h)=%.3f  P(pass | 3 h)=%.3f  P(pass | 6 h)=%.3f"
          % (lam, b0, b1, sigmoid(b0+b1*4.5), sigmoid(b0+b1*3), sigmoid(b0+b1*6)))

# Prediction using the lambda = 0.1 model
r = optimize.minimize(pen_nll, x0=[0., 0.], args=(hours, passed, 0.1), method="BFGS")
b0, b1 = r.x
print("\nFitted model (lambda=0.1): logit(p) = %.4f + %.4f * X" % (b0, b1))
print("Predicted probabilities:", np.round(sigmoid(b0 + b1*hours), 3))

lambda=0.01  beta0=-25.4388 beta1= 5.6531  P(pass | 4.5 h)=0.500  P(pass | 3 h)=0.000  P(pass | 6 h)=1.000
lambda=0.1   beta0=-12.0463 beta1= 2.6770  P(pass | 4.5 h)=0.500  P(pass | 3 h)=0.018  P(pass | 6 h)=0.982
lambda=1.0   beta0= -5.2639 beta1= 1.1698  P(pass | 4.5 h)=0.500  P(pass | 3 h)=0.147  P(pass | 6 h)=0.853

Fitted model (lambda=0.1): logit(p) = -12.0463 + 2.6770 * X
Predicted probabilities: [0.    0.001 0.018 0.208 0.792 0.982 0.999 1.   ]


# LAB 8 – Simulation and Monte Carlo Methods

In [20]:
rng = np.random.default_rng(2026)

### Q1. 10 000 observations from N(50, 10²)

In [21]:
z = rng.normal(loc=50, scale=10, size=10_000)
print("Sample mean = %.4f (theory 50)" % z.mean())
print("Sample sd   = %.4f (theory 10)" % z.std(ddof=1))
print("First 5 values:", z[:5])

Sample mean = 49.9793 (theory 50)
Sample sd   = 10.0567 (theory 10)
First 5 values: [42.0688 52.4057 31.0367 63.9577 56.3829]


### Q2. 10 000 Bernoulli(0.3) observations – sample proportion

In [22]:
bern = rng.binomial(n=1, p=0.3, size=10_000)
print("Number of successes = ", bern.sum())
print("Sample proportion   = %.4f (theory 0.3)" % bern.mean())

Number of successes =  2948
Sample proportion   = 0.2948 (theory 0.3)


### Q3. Monte Carlo estimate of ∫₀¹ x³ dx  (exact = 0.25)

In [23]:
N = 1_000_000
u = rng.uniform(0, 1, N)
vals = u**3
est = vals.mean()
se  = vals.std(ddof=1) / np.sqrt(N)
print("MC estimate = %.5f  (exact 0.25), standard error = %.5f" % (est, se))

MC estimate = 0.24973  (exact 0.25), standard error = 0.00028


### Q4. Estimate π using one million points

In [24]:
N = 1_000_000
x = rng.uniform(-1, 1, N)
y = rng.uniform(-1, 1, N)
inside = (x**2 + y**2) <= 1
pi_hat = 4 * inside.mean()
se = 4 * np.sqrt(inside.mean()*(1-inside.mean())/N)
print("pi estimate = %.5f  (true %.5f), standard error ≈ %.5f" % (pi_hat, np.pi, se))

pi estimate = 3.14146  (true 3.14159), standard error ≈ 0.00164


### Q5. Estimate P(X > 1.96), X ~ N(0, 1)

In [25]:
N = 1_000_000
x = rng.standard_normal(N)
p_hat = (x > 1.96).mean()
se = np.sqrt(p_hat*(1-p_hat)/N)
print("MC estimate = %.5f  (SE %.5f)" % (p_hat, se))
print("Exact value = %.5f" % stats.norm.sf(1.96))

MC estimate = 0.02507  (SE 0.00016)
Exact value = 0.02500


### Q6. Importance sampling for E(X²), X ~ U(0, 1)  (exact = 1/3)

Target density f(x) = 1 on (0, 1). Choose a proposal g(x) = 2x on (0, 1) (a Beta(2, 1) distribution), which puts more weight where x² is large. Sampling: X = √U. Weights w(x) = f(x)/g(x) = 1/(2x). The estimator is mean[ x²·w(x) ] = mean[ x/2 ].

In [26]:
N = 100_000

# plain Monte Carlo
x_plain = rng.uniform(0, 1, N)
plain = x_plain**2

# importance sampling with g(x) = 2x
x_is = np.sqrt(rng.uniform(0, 1, N))       # inverse-CDF sampling from g
w = 1.0 / (2*x_is)                         # f/g
is_vals = x_is**2 * w

print("Exact value             : %.5f" % (1/3))
print("Plain MC estimate       : %.5f   SE = %.6f" % (plain.mean(),   plain.std(ddof=1)/np.sqrt(N)))
print("Importance sampling est.: %.5f   SE = %.6f" % (is_vals.mean(), is_vals.std(ddof=1)/np.sqrt(N)))
print("Variance reduction factor: %.2f" % (plain.var(ddof=1)/is_vals.var(ddof=1)))

Exact value             : 0.33333
Plain MC estimate       : 0.33356   SE = 0.000942
Importance sampling est.: 0.33347   SE = 0.000373
Variance reduction factor: 6.37


### Q7. Importance sampling vs control variates

Both are **variance-reduction** techniques for Monte Carlo estimation, but they work differently.

| | **Importance sampling** | **Control variates** |
|---|---|---|
| Idea | Sample from a *different* distribution g that concentrates on the important region, and correct with weights f/g | Keep sampling from the original distribution, but subtract a correlated quantity whose expectation is known |
| Estimator | (1/N) Σ h(Xᵢ)·f(Xᵢ)/g(Xᵢ), Xᵢ ~ g | (1/N) Σ [h(Xᵢ) − c(Y(Xᵢ) − E[Y])] |
| What is changed | The sampling distribution | The quantity being averaged |
| Requirement | A good g that is easy to sample from, with heavier tails than |h·f| | A control Y highly correlated with h(X) and with exactly known mean |
| Best for | Rare events / tail probabilities | Cases where a simple, similar problem has a known answer |
| Risk | Poor g gives huge or infinite variance | Weak correlation gives little benefit; still unbiased |

Variance reduction in control variates is by a factor of about (1 − ρ²) where ρ is the correlation between h(X) and the control.

In [27]:
# Quick control-variate demo for E(X^2), X~U(0,1): use Y = X with known mean 0.5
N = 100_000
x = rng.uniform(0, 1, N)
h = x**2
c = np.cov(h, x)[0, 1] / np.var(x, ddof=1)       # optimal coefficient
cv = h - c*(x - 0.5)
print("Plain MC          : %.5f   SE = %.6f" % (h.mean(),  h.std(ddof=1)/np.sqrt(N)))
print("Control variate   : %.5f   SE = %.6f" % (cv.mean(), cv.std(ddof=1)/np.sqrt(N)))

Plain MC          : 0.33314   SE = 0.000943
Control variate   : 0.33326   SE = 0.000235
